# 实验四：Kaggle 表情图片分类（CNN + 数据增强）

本 Notebook 对应实验 4 的 Kaggle 任务：读取 `48×48` 灰度人脸图像，使用**在线数据增强**训练 CNN，并预测 7 种表情。

- 训练集：`label, feature`
- 测试集：`id, feature`
- 输出文件：`/kaggle/working/submission.csv`，列名严格为 `ID,Label`
- 推荐加速器：**GPU T4**（若当前 Kaggle 的 PyTorch 不支持 P100，会自动回退 CPU 并给出提示）

数据增强只作用于训练集；验证集和测试集只做标准化。模型输出原始 logits，直接交给 `CrossEntropyLoss`，不在输出层额外添加 Softmax。


In [ ]:
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED = 42
NUM_CLASSES = 7
IMAGE_SIZE = 48
CLASS_NAMES = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral"]


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = torch.cuda.is_available()


def select_device():
    if not torch.cuda.is_available():
        print("CUDA 不可用，将使用 CPU。Kaggle 上建议在 Settings 中选择 GPU T4。")
        return torch.device("cpu")

    gpu_name = torch.cuda.get_device_name(0)
    capability = torch.cuda.get_device_capability(0)
    gpu_arch = f"sm_{capability[0]}{capability[1]}"
    compiled_arches = set(torch.cuda.get_arch_list())
    print("检测到 GPU:", gpu_name, "架构:", gpu_arch)
    print("当前 PyTorch 支持的 CUDA 架构:", sorted(compiled_arches))

    if compiled_arches and gpu_arch not in compiled_arches:
        print(
            f"警告：当前 PyTorch 未包含 {gpu_arch} 的 CUDA 内核，将自动使用 CPU。\n"
            "请在 Kaggle Settings -> Accelerator 中改选 T4，然后 Restart Session 并 Run All。"
        )
        return torch.device("cpu")

    try:
        probe = nn.Conv2d(1, 1, kernel_size=3, padding=1).cuda()
        probe_input = torch.zeros(1, 1, 8, 8, device="cuda")
        _ = probe(probe_input).sum().item()
        del probe, probe_input
        return torch.device("cuda")
    except Exception as error:
        print(
            "CUDA 卷积测试失败，将自动使用 CPU。建议改选 T4 并重启会话。错误类型：",
            type(error).__name__,
        )
        return torch.device("cpu")


seed_everything()
DEVICE = select_device()

OUTPUT_DIR = Path(os.environ.get("LAB4_WORKING_DIR", "/kaggle/working"))
if not OUTPUT_DIR.parent.exists():
    OUTPUT_DIR = Path.cwd()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PyTorch:", torch.__version__)
print("设备:", DEVICE)
print("输出目录:", OUTPUT_DIR)


## 1. 定位并读取竞赛数据

代码会在 Kaggle Input 中递归查找**同一目录下**的 `train.csv` 和 `test.csv`，并用实际列名筛选表情分类数据，因此不依赖竞赛挂载目录的固定名称。


In [ ]:
def normalized_columns(df):
    return {str(column).strip().lower(): column for column in df.columns}


def valid_expression_pair(train_path, test_path):
    try:
        train_head = pd.read_csv(train_path, nrows=2)
        test_head = pd.read_csv(test_path, nrows=2)
    except Exception:
        return False
    train_cols = normalized_columns(train_head)
    test_cols = normalized_columns(test_head)
    pixel_names = {"feature", "pixels", "pixel", "image"}
    return (
        bool(pixel_names.intersection(train_cols))
        and bool(pixel_names.intersection(test_cols))
        and bool({"label", "emotion", "target"}.intersection(train_cols))
    )


def find_input_files():
    roots = []
    env_root = os.environ.get("LAB4_INPUT_ROOT")
    if env_root:
        roots.append(Path(env_root).expanduser())
    roots.extend([
        Path("/kaggle/input/2024-zjutest4"),
        Path("/kaggle/input/2024-zjutest-4"),
        Path("/kaggle/input/competitions/2024-zjutest4"),
        Path("/kaggle/input"),
    ])

    candidate_pairs = []
    visited = set()
    for root in roots:
        if not root.exists():
            continue
        for train_path in sorted(root.rglob("*.csv")):
            if train_path.name.lower() != "train.csv":
                continue
            key = str(train_path.resolve())
            if key in visited:
                continue
            visited.add(key)
            test_path = train_path.parent / "test.csv"
            if test_path.is_file() and valid_expression_pair(train_path, test_path):
                path_text = str(train_path.parent).lower().replace("-", "")
                priority = 0 if "zjutest4" in path_text else 1
                candidate_pairs.append((priority, train_path, test_path))

    if candidate_pairs:
        _, train_path, test_path = sorted(candidate_pairs, key=lambda item: (item[0], str(item[1])))[0]
        return train_path.parent, train_path, test_path

    kaggle_input = Path("/kaggle/input")
    mounted = []
    if kaggle_input.exists():
        mounted = [
            str(path.relative_to(kaggle_input))
            for path in sorted(kaggle_input.rglob("*"))
            if path.is_file()
        ][:30]
    raise FileNotFoundError(
        "没有找到符合 label/feature 与 id/feature 结构的 train.csv、test.csv。"
        "请在 Notebook 右侧 Add Input 中添加本次竞赛数据。"
        f"当前已挂载文件（最多显示 30 个）：{mounted}"
    )


INPUT_DIR, TRAIN_PATH, TEST_PATH = find_input_files()
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("输入目录:", INPUT_DIR)
print("train:", train_df.shape, list(train_df.columns))
print("test :", test_df.shape, list(test_df.columns))


In [ ]:
def pick_column(df, preferred_names, purpose):
    columns = normalized_columns(df)
    for name in preferred_names:
        if name.lower() in columns:
            return columns[name.lower()]
    raise ValueError(f"无法识别{purpose}列；实际列名为 {list(df.columns)}")


def parse_pixels(series, image_size=IMAGE_SIZE):
    expected = image_size * image_size
    rows = []
    for row_index, value in enumerate(series.astype(str)):
        pixels = np.fromstring(value, sep=" ", dtype=np.float32)
        if pixels.size != expected:
            raise ValueError(
                f"第 {row_index} 行含 {pixels.size} 个像素，预期 {expected} 个；"
                "请检查 feature 列和空格分隔格式。"
            )
        rows.append(pixels)
    array = np.stack(rows).reshape(-1, 1, image_size, image_size)
    return array / 255.0


LABEL_COL = pick_column(train_df, ["label", "emotion", "target"], "训练标签")
TRAIN_PIXELS_COL = pick_column(train_df, ["feature", "pixels", "pixel", "image"], "训练像素")
TEST_PIXELS_COL = pick_column(test_df, ["feature", "pixels", "pixel", "image"], "测试像素")

y_all = train_df[LABEL_COL].to_numpy(dtype=np.int64)
X_all = parse_pixels(train_df[TRAIN_PIXELS_COL])
X_test = parse_pixels(test_df[TEST_PIXELS_COL])

observed_labels = sorted(np.unique(y_all).tolist())
if observed_labels != list(range(NUM_CLASSES)):
    raise ValueError(f"训练标签应为 0~6，实际为 {observed_labels}")

data_mean = float(X_all.mean())
data_std = float(X_all.std())
if data_std <= 0:
    raise ValueError("训练图像标准差为 0，无法标准化。")

print("训练图像:", X_all.shape, X_all.dtype, f"范围 [{X_all.min():.3f}, {X_all.max():.3f}]")
print("测试图像:", X_test.shape)
print("类别计数:", np.bincount(y_all, minlength=NUM_CLASSES).tolist())
print(f"训练集 mean={data_mean:.5f}, std={data_std:.5f}")


In [ ]:
fig, axes = plt.subplots(2, 7, figsize=(14, 4.5))
for label in range(NUM_CLASSES):
    indices = np.flatnonzero(y_all == label)[:2]
    for row, index in enumerate(indices):
        axes[row, label].imshow(X_all[index, 0], cmap="gray", vmin=0, vmax=1)
        axes[row, label].set_title(f"{label}: {CLASS_NAMES[label]}")
        axes[row, label].axis("off")
plt.suptitle("Training samples by class")
plt.tight_layout()
plt.show()


## 2. 分层划分与在线数据增强

增强组合包括：

1. 随机水平翻转；
2. 小角度旋转、平移与缩放；
3. 随机亮度和对比度变化；
4. Random Erasing（轻度随机遮挡）。

不使用垂直翻转，因为倒置人脸与真实测试分布差异较大。在线增强会在每个 epoch 产生不同样本，不需要把增强后的图片全部复制到内存中。


In [ ]:
train_idx, val_idx = train_test_split(
    np.arange(len(y_all)),
    test_size=0.15,
    random_state=SEED,
    stratify=y_all,
)

augmentation_only = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(
        degrees=12,
        translate=(0.08, 0.08),
        scale=(0.92, 1.08),
    ),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
])

train_transform = transforms.Compose([
    augmentation_only,
    transforms.Normalize(mean=(data_mean,), std=(data_std,)),
    transforms.RandomErasing(
        p=0.20,
        scale=(0.02, 0.12),
        ratio=(0.5, 2.0),
        value=0.0,
    ),
])
eval_transform = transforms.Normalize(mean=(data_mean,), std=(data_std,))


class FacialExpressionDataset(Dataset):
    def __init__(self, images, labels=None, transform=None):
        self.images = torch.from_numpy(images).float()
        self.labels = None if labels is None else torch.from_numpy(labels).long()
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = self.images[index]
        if self.transform is not None:
            image = self.transform(image)
        if self.labels is None:
            return image
        return image, self.labels[index]


train_set = FacialExpressionDataset(X_all[train_idx], y_all[train_idx], train_transform)
val_set = FacialExpressionDataset(X_all[val_idx], y_all[val_idx], eval_transform)
test_set = FacialExpressionDataset(X_test, transform=eval_transform)

BATCH_SIZE = 256 if DEVICE.type == "cuda" else 64
NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0
loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)
if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True

train_loader = DataLoader(train_set, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_set, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_set, shuffle=False, **loader_kwargs)

print("训练/验证/测试:", len(train_set), len(val_set), len(test_set))
print("batch size:", BATCH_SIZE)


In [ ]:
preview_transform = transforms.Compose([
    augmentation_only,
    transforms.RandomErasing(
        p=0.35,
        scale=(0.02, 0.12),
        ratio=(0.5, 2.0),
        value="random",
    ),
])

sample_index = int(train_idx[0])
sample_image = torch.from_numpy(X_all[sample_index]).float()
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
axes = axes.ravel()
axes[0].imshow(sample_image[0], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Original")
for i in range(1, len(axes)):
    augmented = preview_transform(sample_image.clone())
    axes[i].imshow(augmented[0].clamp(0, 1), cmap="gray", vmin=0, vmax=1)
    axes[i].set_title(f"Augmented {i}")
for axis in axes:
    axis.axis("off")
plt.suptitle("Online data augmentation examples")
plt.tight_layout()
plt.show()


## 3. CNN 模型

空间尺寸遵循实验 PDF：`48→24→12→6`，通道数为 `1→64→128→256`。每个阶段增加第二个 `3×3` 卷积以增强特征提取；分类头使用 `9216→1024→256→7`，比课件中的超大 FC 层更节省显存，也更不容易过拟合。


In [ ]:
class ConvStage(nn.Sequential):
    def __init__(self, in_channels, out_channels, dropout):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.RReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.RReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(p=dropout),
        )


class ExpressionCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            ConvStage(1, 64, dropout=0.05),       # (N, 1, 48, 48) -> (N, 64, 24, 24)
            ConvStage(64, 128, dropout=0.10),     # -> (N, 128, 12, 12)
            ConvStage(128, 256, dropout=0.15),    # -> (N, 256, 6, 6)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.35),
            nn.Linear(256 * 6 * 6, 1024),
            nn.BatchNorm1d(1024),
            nn.RReLU(inplace=True),
            nn.Dropout(p=0.50),
            nn.Linear(1024, 256),
            nn.RReLU(inplace=True),
            nn.Dropout(p=0.25),
            nn.Linear(256, num_classes),
        )
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Conv2d):
            nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
        elif isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, x):
        return self.classifier(self.features(x))


model = ExpressionCNN().to(DEVICE)
with torch.no_grad():
    shape_check = model(torch.zeros(2, 1, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE))
assert shape_check.shape == (2, NUM_CLASSES)

trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(model)
print(f"可训练参数: {trainable_params:,}")
print("形状检查:", tuple(shape_check.shape))


## 4. 训练与保存最佳模型

训练使用轻度类别权重、label smoothing、AdamW、余弦退火、AMP、梯度裁剪和早停。保存标准 `state_dict`，验证准确率最高的权重将用于最终预测。


In [ ]:
class_counts = np.bincount(y_all[train_idx], minlength=NUM_CLASSES)
class_weights = np.sqrt(class_counts.mean() / class_counts)
class_weights = np.clip(class_weights / class_weights.mean(), 0.60, 2.00)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=2e-4)

EPOCHS = 35
PATIENCE = 8
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-5,
)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))


def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            if training:
                optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                logits = model(images)
                loss = criterion(logits, labels)

            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                scaler.step(optimizer)
                scaler.update()

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_samples += batch_size

    return total_loss / total_samples, total_correct / total_samples


BEST_PATH = OUTPUT_DIR / "best_lab4_expression_cnn.pth"
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = -1.0
stale_epochs = 0

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss, val_acc = run_epoch(val_loader, training=False)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    improved = val_acc > best_val_acc
    if improved:
        best_val_acc = val_acc
        stale_epochs = 0
        torch.save(model.state_dict(), BEST_PATH)
    else:
        stale_epochs += 1

    lr = optimizer.param_groups[0]["lr"]
    print(
        f"[{epoch:02d}/{EPOCHS}] {time.time() - start:6.1f}s "
        f"train loss={train_loss:.4f} acc={train_acc:.4f} | "
        f"val loss={val_loss:.4f} acc={val_acc:.4f} | lr={lr:.2e}"
        + ("  <- saved" if improved else "")
    )

    if stale_epochs >= PATIENCE:
        print(f"验证准确率连续 {PATIENCE} 轮未提升，提前停止。")
        break

assert BEST_PATH.is_file(), "最佳模型文件未生成"
print(f"最佳验证准确率: {best_val_acc:.4f}")
print("最佳权重:", BEST_PATH)


In [ ]:
epochs_ran = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_ran, history["train_loss"], label="train")
axes[0].plot(epochs_ran, history["val_loss"], label="validation")
axes[0].set(title="Loss", xlabel="Epoch", ylabel="Cross entropy")
axes[0].legend()

axes[1].plot(epochs_ran, history["train_acc"], label="train")
axes[1].plot(epochs_ran, history["val_acc"], label="validation")
axes[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1))
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
try:
    best_state = torch.load(BEST_PATH, map_location=DEVICE, weights_only=True)
except TypeError:
    best_state = torch.load(BEST_PATH, map_location=DEVICE)
model.load_state_dict(best_state)
model.eval()

val_true, val_pred = [], []
with torch.no_grad():
    for images, labels in val_loader:
        logits = model(images.to(DEVICE, non_blocking=True))
        val_pred.extend(logits.argmax(dim=1).cpu().numpy())
        val_true.extend(labels.numpy())

print(classification_report(
    val_true,
    val_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
))

cm = confusion_matrix(
    val_true,
    val_pred,
    labels=list(range(NUM_CLASSES)),
    normalize="true",
)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title("Normalized validation confusion matrix")
plt.tight_layout()
plt.show()


## 5. 测试集预测与生成提交文件

使用原图和水平翻转图的 logits 均值进行 TTA。提交 ID 直接来自 `test.csv`，并在写出后重新读取，检查列名、行数、ID 顺序和标签范围。


In [ ]:
model.eval()
test_predictions = []
with torch.no_grad():
    for images in test_loader:
        images = images.to(DEVICE, non_blocking=True)
        logits_original = model(images)
        logits_flipped = model(torch.flip(images, dims=[3]))
        logits = (logits_original + logits_flipped) / 2
        test_predictions.extend(logits.argmax(dim=1).cpu().numpy())

test_columns = normalized_columns(test_df)
test_id_column = test_columns.get("id")
test_ids = (
    test_df[test_id_column].to_numpy()
    if test_id_column is not None
    else np.arange(len(test_df))
)

submission = pd.DataFrame({
    "ID": test_ids,
    "Label": np.asarray(test_predictions, dtype=np.int64),
})

SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

reloaded = pd.read_csv(SUBMISSION_PATH)
assert list(reloaded.columns) == ["ID", "Label"]
assert len(reloaded) == len(test_df)
assert reloaded["ID"].equals(pd.Series(test_ids, name="ID"))
assert reloaded["Label"].between(0, NUM_CLASSES - 1).all()

display(reloaded.head(10))
print("提交文件:", SUBMISSION_PATH)
print("行数:", len(reloaded))
print("标签分布:", reloaded["Label"].value_counts().sort_index().to_dict())


## 6. 提交

运行完全部单元后，在 Kaggle Notebook 的 **Output** 中找到 `submission.csv`，再点击 **Submit to Competition**。最终提交文件必须有 6711 条预测（以实际 `test.csv` 行数为准），列名严格为 `ID,Label`。
